# Load a Data Vault satellite: TransactionHistory (local)

Loads `TransactionHistory`'s attributes into a Data Vault satellite on a local Spark session, from
a Parquet file through a DQD, with the pipeline's steps composed one by one. The satellite only
ever takes new rows: one per change, delete or reappearance of a key's attributes, never an
update. Everything the load needs is one JSON load context; every section and field is described
in [docs/reference/load-context.md](../../docs/reference/load-context.md), and the task in
[docs/guides/load-a-data-vault-entity.md](../../docs/guides/load-a-data-vault-entity.md).

The hub is not loaded here. In a real vault it is loaded first, by `dv_hub_pipeline`, so the
satellite never holds a key its hub lacks; see
[NB_BI_LoadEntity_DataVault_Gen3.ipynb](NB_BI_LoadEntity_DataVault_Gen3.ipynb) for both.

The notebook runs top to bottom:

1. **Input parameters**: the run switches and the load context.
2. **Library**: `spark_quick_start` from this repository's `src/`, with no install.
3. **Imports**.
4. **Spark session**: local, with Delta.
5. **Pipeline**: built from the switches and run; the result is the cell's output.

It expects a kernel with PySpark and `delta-spark`, started anywhere inside the repository.
The same notebook runs on macOS, Linux and Windows; on Windows it also needs `HADOOP_HOME`
pointing at a Hadoop folder whose `bin/` holds `winutils.exe` and `hadoop.dll`.


## 1. Input parameters

In [ ]:
from pathlib import Path

# The repository root: the nearest folder at or above the working directory holding pyproject.toml.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())


def repo_path(*parts: str) -> str:
    """A path inside the repository, as the load context expects it."""
    return REPO_ROOT.joinpath(*parts).as_posix()


def struct(*fields: tuple[str, str]) -> dict:
    """A Spark StructType as JSON, every field nullable."""
    return {"type": "struct",
            "fields": [{"name": n, "type": t, "nullable": True, "metadata": {}} for n, t in fields]}

In [ ]:
# Run switches
LOAD_TYPE = "Incremental"  # "Full" also records every key absent from the source as deleted
HASH_ALGORITHM = "sha256"  # or "md5"; sealed on the satellite at its first write
DRIVER_MEMORY = "1g"       # Spark's default; raise it for a large load (backlog F153)
SHUFFLE_PARTITIONS = 200

SOURCE_PARQUET = repo_path("data", "Production_TransactionHistory",
                           "part-00000-f8e129a4-417b-4815-bb81-9c4e9b4b0770-c000.snappy.parquet")
ATTRIBUTES = ["ProductID", "ReferenceOrderID", "ReferenceOrderLineID", "TransactionDate",
              "TransactionType", "Quantity", "ActualCost", "ModifiedDate"]

# The load context: the five required sections and a dqd_context source. The hash key is built
# upstream, in the DQD: trim, upper-case, sha2 256, the same expression the hub's context uses.
# Op is the source's delete signal, "N" here; MarkRowAsDeleted below sets it to "Y" for one row.
# hwm_context reads only the rows modified since hwm_value; a Full load reads everything, since
# a Full load with a watermark would delete every key the watermark left out, and is refused.
load_context_dict = {
    "global_context": {
        "source_system": "CS",
        "id": "TransactionHistory_cs_s",
        "entity_name": "TransactionHistory",
        "staging_table_path": repo_path("data", "Staging", "Production_TransactionHistory"),
        "target_table_path": repo_path("data", "Target", "Vault", "TransactionHistory_cs_s"),
    },
    "scd_context": {
        "scd_type": "dv_satellite",
        "business_key_column": "HashKey",
        "load_type": LOAD_TYPE,
        "hash_algorithm": HASH_ALGORITHM,
        "source_deleted_flag_column": "Op",
        "deleted_flag_value": "Y",
        "exclude_from_hash": ["Quantity"],
    },
    "runtime_context": {
        "application_name": "VaultLoad",
        "correlation_id": "vault-local",
        "pipeline_name": "TransactionHistory_cs_s",
    },
    "logging_context": {
        "log_file_dir": repo_path("data", "Logs"),
        "logging_level": "INFO",
        "log_file_name": "Vault_TransactionHistory_cs_s",
        "verbose": False,
    },
    **({"hwm_context": {"hwm_column_name": "ModifiedDate", "hwm_value": "2014-08-01 00:00:00"}}
       if LOAD_TYPE == "Incremental" else {}),
    "dqd_context": {
        "source_kind": "parquet",
        "source_identifier": SOURCE_PARQUET,
        "transformations": [
            {"name": "HashKey", "expression": {"function": "sha2", "arguments": [
                {"function": "upper", "arguments": [{"function": "trim", "arguments": [
                    {"function": "coalesce", "arguments": [
                        {"cast": {"expression": {"column": "TransactionID"}, "data_type": "string"}},
                        {"literal": ""}]}]}]},
                {"literal": 256}]}},
        ],
        "selection": [
            {"name": "HashKey"},
            *({"name": c} for c in ATTRIBUTES),
            {"name": "Op", "expression": {"literal": "N"}},
        ],
    },
}


## 2. Library

In [ ]:
import sys

# The library as a source copy (./src on sys.path), the way it also ships: nothing to install.
# Put first, so an installed copy cannot shadow it. Re-running this cell and the imports cell
# picks up edits to src/ without restarting the kernel.
src = str(REPO_ROOT / "src")
if src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
for name in [m for m in sys.modules if m == "spark_quick_start" or m.startswith("spark_quick_start.")]:
    del sys.modules[name]

import spark_quick_start

print(f"spark_quick_start from {Path(spark_quick_start.__file__).parent}")

## 3. Imports

In [ ]:
import hashlib
import json
import os
import platform
from importlib.metadata import version

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

from spark_quick_start.adapters import LocalAdapter
from spark_quick_start.orchestration import LoadOrchestrator
from spark_quick_start.orchestration.transformers import (
    DQDTransformer,
    DVSatelliteChangeDetectorTransformer,
    DVSatelliteIncrementalTargetTransformer,
    DVSatelliteTargetTransformer,
    DVSatelliteWriterTransformer,
    MetricsChartTransformer,
    ProcessTransformer,
    StagedKeyGuardTransformer,
)


## 4. Spark session

In [ ]:
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName(load_context_dict["runtime_context"]["application_name"])
    .master("local[*]")
    # Local mode runs the driver and every task in one JVM, sharing this heap. It is fixed when
    # the JVM starts: a new value needs a kernel restart, not only a new session.
    .config("spark.driver.memory", DRIVER_MEMORY)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

if os.name == "nt":
    # Windows only: Spark reaches the local file system through Hadoop's winutils.exe and
    # hadoop.dll, found in HADOOP_HOME/bin, and must not try Hadoop's native Unix libraries.
    hadoop_home = os.environ.get("HADOOP_HOME")
    if not hadoop_home:
        raise EnvironmentError("Set HADOOP_HOME to a folder whose bin/ holds winutils.exe and hadoop.dll.")
    hadoop_bin = os.path.join(hadoop_home, "bin")
    if hadoop_bin not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")
    builder = (
        builder
        .config("spark.hadoop.io.native.lib.available", "false")
        .config("spark.driver.extraLibraryPath", hadoop_bin)
        .config("spark.executor.extraLibraryPath", hadoop_bin)
    )

# delta-spark adds the Delta package that matches its own version and this Spark's Scala build
# (delta-spark_2.12 on Spark 3.5, delta-spark_4.x_2.13 on Spark 4), from the Ivy cache when present.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")

print(f"Spark {spark.version} on {platform.system()}, delta-spark {version('delta-spark')}, "
      f"driver memory {spark.sparkContext.getConf().get('spark.driver.memory', '1g')}")

## 5. Pipeline

### Optional: steps for trying out changes

Not part of the pipeline below. Add one after `StagedKeyGuardTransformer`, for example
`.add(PatchTransformer(hash_key(213376), "ActualCost", 99.0))` to see a `changed` row, or
`.add(MarkRowAsDeleted(hash_key(213376)))` to see a `deleted` row that carries the latest
attributes forward. Removing it again on the next run shows the key reappear. A change to
`Quantity` alone writes nothing: it is left out of the hash diff. `hash_key` rebuilds the DQD's
hash key in Python, so the key can be named by its `TransactionID`.
Setting `spark.driver.memory` only takes effect on a new session: restart the kernel after
changing `DRIVER_MEMORY`.


In [ ]:
def hash_key(transaction_id) -> str:
    """The DQD's HashKey for one TransactionID: sha256 hex of its trimmed, upper-cased text."""
    return hashlib.sha256(str(transaction_id).strip().upper().encode("utf-8")).hexdigest()


class MarkRowAsDeleted(ProcessTransformer):
    """Set the source's delete signal on the row whose HashKey is key_value."""

    def __init__(self, key_value: str) -> None:
        self._key_value = key_value

    def _run(self, state, ctx):
        s = ctx.scd_ctx
        signal = s.source_deleted_flag_column
        return state.with_df(state.df.withColumn(signal, F.when(
            F.col(s.business_key_column) == F.lit(self._key_value), F.lit(s.deleted_flag_value)
        ).otherwise(F.col(signal))))


class PatchTransformer(ProcessTransformer):
    """Overwrite one cell of the row whose HashKey is key_value, keeping the column's type."""

    def __init__(self, key_value: str, column_name: str, new_value) -> None:
        self._key_value, self._column_name, self._new_value = key_value, column_name, new_value

    def _run(self, state, ctx):
        key, df = ctx.scd_ctx.business_key_column, state.df
        data_type = df.schema[self._column_name].dataType
        return state.with_df(df.withColumn(self._column_name, F.when(
            F.col(key) == F.lit(self._key_value), F.lit(self._new_value).cast(data_type)
        ).otherwise(F.col(self._column_name))))


In [ ]:
adapter = LocalAdapter(base_path=repo_path())
FULL = LOAD_TYPE == "Full"

result = (
    LoadOrchestrator(json.dumps(load_context_dict), spark, adapter=adapter)
    .add(DQDTransformer())
    .add(StagedKeyGuardTransformer())                        # a key staged twice, NULL or empty
    .add_if(FULL, DVSatelliteTargetTransformer())            # every key's latest row
    .add_if(not FULL, DVSatelliteIncrementalTargetTransformer())  # the batch's keys only
    .add(DVSatelliteChangeDetectorTransformer())             # new, changed, deleted
    .add(DVSatelliteWriterTransformer())                     # insert-only MERGE on (HashKey, LoadTS)
    .add(MetricsChartTransformer())
).run()

result


### Inspect the satellite: each key's current row

In [ ]:
# Open-source Spark 3.5 has no QUALIFY, so the current row is picked in a subquery, as on Fabric.
satellite: DataFrame = spark.read.format("delta").load(load_context_dict["global_context"]["target_table_path"])
satellite.createOrReplaceTempView("satellite")
spark.sql("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY HashKey ORDER BY LoadTS DESC) AS rn FROM satellite
    ) AS s WHERE rn = 1
""").drop("rn").orderBy(F.col("LoadTS").desc()).show(5, truncate=False)


### Inspect one key's history

In [ ]:
from delta.tables import DeltaTable

# One key's whole history, oldest first: a row per change, delete or reappearance.
# satellite.filter(F.col("HashKey") == hash_key(213376)).orderBy("LoadTS") \
#     .select("LoadTS", "IsDeleted", "ActualCost", "Quantity", "HashDiff", "RecordSource").show(truncate=False)

# satellite.filter(F.col("HashKey") == "ecf04ee677ffec189c404c3d04e0f223ed0af6b8c1d02ee4473d56a645d8ea4b").orderBy("LoadTS") \
#     .select("HashKey", "LoadTS", "IsDeleted", "ActualCost", "Quantity", "HashDiff", "RecordSource").show(truncate=False)


# Experiment, not part of the reference load: it rewrites a historical row of an insert-only
# satellite, to see how the detector treats a tampered IsDeleted flag. It names one row of the
# owner's local run (that LoadTS exists nowhere else). Do not run it on a satellite you keep.
# DeltaTable.forPath(
#     spark, load_context_dict["global_context"]["target_table_path"]
# ).update(
#     condition=(
#         (F.col("HashKey") == "ecf04ee677ffec189c404c3d04e0f223ed0af6b8c1d02ee4473d56a645d8ea4b")
#         & (F.col("LoadTS") == F.lit("2026-10-01 19:38:44.027657").cast("timestamp"))
#     ),
#     set={"IsDeleted": F.lit("Y")},
# )
